# Analyse Exploratoire des Données — Cancer de la Prostate

**Projet TP Data Science — Msc. Osias Noël TOSSOU**

Ce notebook documente l'EDA complète du dataset `prostate.csv` (97 patients, 9 variables biologiques, cible `lpsa` = log du PSA).

## Plan
1. Contexte et objectifs
2. Description du dataset
3. Qualité des données (valeurs manquantes, doublons, types)
4. Statistiques descriptives
5. Distributions des variables
6. Matrice de corrélation
7. Relation variables / cible
8. Variables catégorielles (svi, gleason)
9. Premières hypothèses métier

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Recherche du fichier de données


df = pd.read_csv("prostate.csv")
df.head()

,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa,train
0,-0.579818,2.769459,50,-1.386294,0,-1.386294,6,0,-0.430783,True
1,-0.994252,3.319626,58,-1.386294,0,-1.386294,6,0,-0.162519,True
2,-0.510826,2.691243,74,-1.386294,0,-1.386294,7,20,-0.162519,True
3,-1.203973,3.282789,58,-1.386294,0,-1.386294,6,0,-0.162519,True
4,0.751416,3.432373,62,-1.386294,0,-1.386294,6,0,0.371564,True


## 1. Contexte et objectifs

Le cancer de la prostate est un enjeu majeur de santé publique masculine, en hausse en Afrique subsaharienne. Le dosage sanguin du PSA est un marqueur clé mais sa disponibilité est limitée dans les centres de premier recours.

**Objectif** : prédire le log(PSA) à partir des variables de biopsie pour orienter le clinicien.

**Type de problème** : régression supervisée.

## 2. Description du dataset

In [4]:
print(f'Lignes : {len(df)}')
print(f'Colonnes : {list(df.columns)}')
print('\nDescription des variables :')
for k, v in FEATURE_DESCRIPTIONS.items():
    print(f'  - {k:<10} : {v}')

Lignes : 97
Colonnes : ['lcavol', 'lweight', 'age', 'lbph', 'svi', 'lcp', 'gleason', 'pgg45', 'lpsa', 'train']

Description des variables :


NameError: name 'FEATURE_DESCRIPTIONS' is not defined

## 3. Qualité des données

In [ ]:
print('Valeurs manquantes :')
print(df.isna().sum())
print(f'\nDoublons : {df.duplicated().sum()}')
print(f'\nTypes :')
print(df.dtypes)

## 4. Statistiques descriptives

In [ ]:
df[FEATURES + [TARGET]].describe().T

## 5. Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
axes = axes.flatten()
for i, col in enumerate(FEATURES + [TARGET]):
    axes[i].hist(df[col], bins=15, color='#3b6fb0', edgecolor='white', alpha=0.85)
    axes[i].set_title(col)
fig.suptitle('Distribution des variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_distributions.png', dpi=110)
plt.show()

## 6. Matrice de corrélation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
corr = df[FEATURES + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, ax=ax)
ax.set_title('Matrice de corrélation', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_correlation_heatmap.png', dpi=110)
plt.show()
corr[TARGET].sort_values(ascending=False)

## 7. Relation variables / cible

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
axes = axes.flatten()
for i, col in enumerate(FEATURES):
    axes[i].scatter(df[col], df[TARGET], alpha=0.6, color='#2c7fb8', edgecolor='white')
    z = np.polyfit(df[col], df[TARGET], 1)
    xs = np.linspace(df[col].min(), df[col].max(), 50)
    axes[i].plot(xs, np.polyval(z, xs), '--', color='#d95f0e')
    axes[i].set_xlabel(col); axes[i].set_ylabel('lpsa')
fig.delaxes(axes[8])
plt.tight_layout()
plt.savefig('../reports/figures/03_scatter_vs_target.png', dpi=110)
plt.show()

## 8. Variables catégorielles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, col in zip(axes, ['svi', 'gleason']):
    sns.boxplot(x=col, y=TARGET, data=df, ax=ax, palette='Set2', order=sorted(df[col].unique()))
    ax.set_title(f'lpsa selon {col}')
plt.tight_layout()
plt.savefig('../reports/figures/04_boxplots_categorical.png', dpi=110)
plt.show()

## 9. Premières hypothèses métier

1. **`lcavol` (volume tumoral)** est la variable la plus corrélée à `lpsa` (r ≈ 0.66). Plus le volume tumoral augmente, plus le PSA est élevé — cohérent cliniquement.
2. **`svi = 1`** (invasion des vésicules séminales) est associé à un lpsa médian plus élevé → marqueur de gravité.
3. **`gleason`** ne montre pas de relation linéaire forte dans ce dataset, mais son effet est visible via `pgg45`.
4. **`age`** corrélation modérée : patients plus âgés → lpsa légèrement plus élevé.
5. **Variables déjà log-transformées** (`lcavol`, `lweight`, `lbph`, `lcp`, `lpsa`) : éviter de re-transformer.

### Conclusion pour la modélisation

- Problème de régression linéaire avec régularisation pertinent (Lasso pour sélection).
- Tester aussi des modèles non-linéaires (RandomForest, GradientBoosting, SVR) pour capturer d'éventuelles interactions.
- Standardisation nécessaire pour les modèles régularisés et SVR.
- Cross-validation impérative : dataset de petite taille (n=97).